# MGP API


see https://www.mathgenealogy.org:8000/api/v2/MGP/

In [ ]:
from mgp import *

In [ ]:
# Log in to the MGP API
# Credentials are stored in credentials.json (git-ignored), not hard-coded here

with open('credentials.json') as f:
    authdata = json.load(f)
token = login(authdata)

In [ ]:
# Fetch all MGP IDs

querydata = None
endpoint = '/api/v2/MGP/acad/all'
acad = json.loads(doquery(endpoint,token,querydata))

In [ ]:
max(acad)

In [ ]:
# Search for all mathematicians in a given ID range
# Cannot exceed 1 minute

querydata = {'start': '1000', 
             'stop' : '1010', 
             'step' : '1'}
endpoint = '/api/v2/MGP/acad/range'
acad = json.loads(doquery(endpoint,token,querydata))

In [ ]:
acad

In [ ]:
# Preparing to crawl data
# Only keeping the information I need

def flatten_acad(data):
    flattened = []
    for record in data:
        acad = record["MGP_academic"]
        degrees = acad["student_data"]["degrees"]
        for degree in degrees:
            flattened.append({
                "ID": acad["ID"],
                "Family Name": acad["family_name"],
                "Given Name": acad["given_name"],
                "Other Names": acad["other_names"],
                "Advised By": ", ".join(f"{k}" for k, v in degree["advised by"].items()),
                "Degree Year": degree["degree_year"],
                "Schools": degree["schools"],
            })
    return flattened

In [ ]:
# Data crawling function

import csv
import os
import time

def write_to_csv(start=0, stop=100, file_path='acad.csv'):
    # Check if the file already exists; raise an error if it does
    if os.path.exists(file_path):
        raise FileExistsError(f"The file '{file_path}' already exists. Please choose a different file path or remove the existing file.")
    
    # Validate query range
    max_queries = 100

    # Initialize headers_exist to False (no headers in a new file)
    headers_exist = False
    
    # Initialize login tracking
    last_login_time = 0  # Stores the last login timestamp
    token = None  # Token for API authentication

    # Auth data for login (loaded from git-ignored credentials.json)
    with open('credentials.json') as f:
        authdata = json.load(f)

    # Process queries in chunks if necessary
    for chunk_start in range(start, stop, max_queries):

        # Log in if more than an hour has passed since the last login
        current_time = time.time()
        if token is None or (current_time - last_login_time) >= 3600:
            print("Logging in...")
            token = login(authdata)  # Assumes `login` function is defined
            last_login_time = current_time

        chunk_stop = min(chunk_start + max_queries, stop)  # Ensure the stop does not exceed the desired range
        querydata = {
            'start': chunk_start,
            'stop': chunk_stop,
            'step': 1
        }

        endpoint = '/api/v2/MGP/acad/range'
        acad = json.loads(doquery(endpoint, token, querydata))  # Assumes `doquery` is defined
        flattened = flatten_acad(acad)  # Assumes `flatten_acad` is defined

        # Append data to the file
        with open(file_path, "a", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=flattened[0].keys(), delimiter='\t')
            if not headers_exist:  # Write headers only for the first chunk
                writer.writeheader()
                headers_exist = True
            writer.writerows(flattened)
        print(f"Processed queries from {chunk_start} to {chunk_stop - 1}.")

In [ ]:
# write_to_csv(start=0, stop=300000, file_path='acad300k_schools_g.csv')